## Tutorial Quickstart - obsługa modeli inżynierskich

Tutorial demonstruje pełny cykl analizy na obiekcie $G(s) = \frac{1}{10s+1}$. Celem tutorialu jest zaprezentowanie prostoty użycia kodu do przeanalizowania modelu inżynierskiego przy pomocy narzędzia jakim jest biblioteka OmniXAI poszerzona o metody SHAP i LIME obsługujące zastosowania inżynierskie.

### Krok 1 instalacja

In [1]:
pip install omnixai python-control scipy numpy matplotlib lime

ERROR: Could not find a version that satisfies the requirement python-control (from versions: none)
ERROR: No matching distribution found for python-control
Note: you may need to restart the kernel to use updated packages.


### Krok 2 - Definicja układu i funkcja predykcyjna

Do tutorialu wybrany został obiekt $G(s) = \frac{1}{10s+1}$, gdyż jest prosty i pozwala na zminimalizowanie czasu symulacji na potrzeby szybkiego tutorialu.

In [2]:
from omnixai.wrapper import ControlSystemWrapper
import numpy as np
import control as ct


# Prosty model: G(s) = 1 / (10s + 1)
G = ct.tf([1], [10, 1])

def predict_mse(params: np.ndarray) -> np.ndarray:
    """
    Funkcja predykcyjna dla SHAP i LIME.
    Wejscie:  (n, 3) - kolumny [K_p, Ki, Kd]
    Wyjscie:  (n,)   - MSE dla kazdej konfiguracji
    """
    results = []
    for K_p, Ki, Kd in params:
        if K_p <= 0 or Ki <= 0 or Kd < 0:
            results.append(1e6)
            continue
        try:
            C = ct.tf([Kd, K_p, Ki], [1, 0])
            T = ct.feedback(C * G)
            t = np.linspace(0, 50, 500)
            y = ControlSystemWrapper(T).predict(np.ones_like(t), t=t)
            results.append(float(np.mean((1.0 - y) ** 2)))
        except Exception:
            results.append(1e6)
    return np.array(results)

### Krok 3 - Dane tła i wyjaśnienia SHAP
Wartość bazowa $\mathbb{E}[f(\mathbf{X})]$ wyznaczana jest automatycznie jako średnia MSE po 30 próbkach tła.

In [3]:
from omnixai.explainers.engineering import ShapEngineering

# Ustawienie ziarna dla powtarzalności wyników
np.random.seed(42)

# Dane tła
background = np.random.uniform(
    low=[1.0, 0.1,  5.0],
    high=[5.0, 1.0, 20.0],
    size=(30, 3))

mse_bg = predict_mse(background)
print(f"E[f(X)] = {mse_bg.mean():.4f}  "
      f"(zakres: [{mse_bg.min():.4f}, {mse_bg.max():.4f}])")

shap_exp = ShapEngineering(
    predict_fn=predict_mse,
    background_data=background,
    feature_names=['K_p', 'Ki', 'Kd'])

test = np.array([[3.0, 0.5, 12.0]])

# Analiza SHAP na danych test
shap_explanation = shap_exp.explain(test)

exp = shap_explanation.get_explanations(index=0)
mse_test = predict_mse(test)[0]

print(f"\nMSE badanej konfiguracji: {mse_test:.4f}")
print(f"Roznica (MSE - E[f]): {mse_test - mse_bg.mean():+.4f}")

print("\nWartosci Shapleya:")
for feat, val, phi in zip(
        exp['features'], exp['values'], exp['scores']):
    print(f"  phi_{feat} = {phi:+.4f}  "
          f"(wartosc: {val:.2f})")

E[f(X)] = 0.0210  (zakres: [0.0083, 0.0438])


  0%|          | 0/1 [00:00<?, ?it/s]


MSE badanej konfiguracji: 0.0165
Roznica (MSE - E[f]): -0.0044

Wartosci Shapleya:
  phi_K_p = -0.0020  (wartosc: 3.00)
  phi_Ki = -0.0019  (wartosc: 0.50)
  phi_Kd = -0.0005  (wartosc: 12.00)


Suma wartości Shapleya wynosi $-0.0020 - 0.0019 - 0.0005 = -0.0004$, a oczekiwana różnica  $MSE - \mathbb{E}[f(\mathbf{X})] = 0.0165 - 0.0210 = -0.0045$. Pewna rozbieżność wynika z aproksymacji Kernel SHAP na małej próbce tła (30 próbek) (przy zwiększeniu liczby próbek zbieżność poprawi się). Przy analizie wpływu parametrów ważna jest jednak hierarchia ich wartości bezwzględnych. $K_p$ odpowiada za największą część łącznego efektu.

### Krok 4 - Wyjaśnienia LIME

In [4]:
from omnixai.explainers.engineering import LIMEEngineering

lime_exp = LIMEEngineering(
    predict_fn=predict_mse,
    background_data=background,
    feature_names=['K_p', 'Ki', 'Kd'],
    mode="regression")

# Analiza LIME na tych samych danych test co dla SHAP
lime_explanation = lime_exp.explain(test, num_samples=50)

exp_l = lime_explanation.get_explanations(index=0)
scores = [abs(w) for w in exp_l['scores']]
max_score = max(scores)

print("Gradienty LIME:")
for feat, val, w in zip(
        exp_l['features'], exp_l['values'], exp_l['scores']):
    kierunek = "zwieksz -> MSE spada" if w < 0 \
        else "zwieksz -> MSE rosnie"
    print(f"  w_{feat} = {w:+.4f}  ({kierunek})")

# Ostrzezenie o potencjalnym artefakcie granicy stabilnosci
if max_score > 10:
    print("\nOSTRZEZENIE: bardzo duze gradienty LIME.")
    print("Sprawdz marginesy stabilnosci: ct.margin(L)")

Gradienty LIME:
  w_K_p = -0.0055  (zwieksz -> MSE spada)
  w_Ki = -0.0054  (zwieksz -> MSE spada)
  w_Kd = -0.0025  (zwieksz -> MSE spada)


Wartości gradientów LIME wskazują, że zwiększenie wszystkich parametrów regulatora PID w analizowanym punkcie lokalnie prowadzi do zmniejszenia wartości MSE, przy czym największy wpływ mają parametry $K_p$ oraz $K_i$, a najmniejszy $K_d$.

### Krok 5 - Dashboard
Po analizie metod LIME i SHAP należy utworzyć obiekt `Dashboard` a następnie wywołać metodę `.show()`. Po wywołaniu metody dashboard będzie dostępny pod adresem http://127.0.0.1:8050.

In [6]:
from omnixai.visualization.dashboard import Dashboard
from omnixai.data.tabular import Tabular

instances = Tabular(test, feature_columns=['K_p', 'Ki', 'Kd'])

Dashboard(
    instances=instances,
    local_explanations={
        "SHAP - wplyw parametrow na MSE": shap_explanation,
        "LIME - lokalne gradienty MSE": lime_explanation,
    }).show()
# http://127.0.0.1:8050

Wyświetlone wyniki na dashboardzie umożliwiają jednoczesną analizę wkładu parametrów metodą SHAP oraz sugestię poprawy nastaw dzięki lokalnym gradientom LIME. Analiza ułatwia interpretację wpływu nastaw PID na jakość regulacji.

Po wykonaniu kroków 1-5 użytkownik dysponuje działającym środowiskiem analizy XAI dla przykładowego układu sterowania. Czas wykonania tutorialu wynosi 8-12 minut (dominuje czas SHAP zależny od liczby próbek tła i szybkości symulatora).